In [12]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import re

In [19]:
file_path = "../data/preprocessed/preprocessed_en-2020-pssd-compendium.csv"
df = pd.read_csv(file_path)

In [20]:
df = df[["Job title", "Salary"]].dropna()   # simplifying, only keeping job title and salary for now

In [21]:
# cleaning salary (maybe we should do this in preprocessing)
df["Salary"] = (
    df["Salary"]
    .astype(str)
    .str.replace(r"[^0-9.]", "", regex=True)
    .astype(float)
)

In [22]:
def tokenize(title):
    title = re.sub(r"[^a-zA-Z0-9, ]+", "", title.lower())
    return title.split()

In [23]:
vocab = sorted(list({w for title in df["Job title"] for w in tokenize(title)}))
stoi = {ch: i + 1 for i, ch in enumerate(vocab)}    # string to int, use 0s for padding
itos = {i: ch for ch, i in stoi.items()}    # int to string (inverse)

In [24]:
def encode(title, max_len=64):
    ids = [stoi.get(c, 0) for c in title.lower()[:max_len]]
    return ids + [0] * (max_len - len(ids))     # padding

In [29]:
class SalaryDataset(Dataset):
    def __init__(self, df):
        self.x = torch.tensor([encode(t) for t in df["Job title"]])
        self.y = torch.tensor(df["Salary"].values, dtype=torch.float32)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.x[i], self.y[i]

In [30]:
dataset = SalaryDataset(df)
loader = DataLoader(dataset, batch_size=2, shuffle=True)

In [31]:
class SalaryTransformer(nn.Module):
    def __init__(self, vocab_size, d_model=64, nhead=4, num_layers=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size + 1, d_model)  # 0 used for padding so vocab_size + 1
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.regressor = nn.Linear(d_model, 1)  # regression
    def forward(self, x):
        emb = self.embedding(x).transpose(0, 1)  # seq_len x batch x d_model
        encoded = self.transformer(emb)
        pooled = encoded.mean(dim=0)  # batch x d_model
        return self.regressor(pooled).squeeze(-1)

In [32]:
model = SalaryTransformer(len(vocab))
criterion = nn.MSELoss()    # mean squared error loss
optimizer = optim.Adam(model.parameters(), lr=1e-3)     # using Adam can change to whatever

c:\Users\Daniyaal\miniconda3\Lib\site-packages\torch\nn\modules\transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


In [33]:
for epoch in range(200):
    for x, y in loader:
        pred = model(x)
        loss = criterion(pred, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    if epoch % 20 == 0:
        print(f"Epoch {epoch}, loss={loss.item():.2f}")

KeyboardInterrupt: 

In [ ]:
with torch.no_grad():
    test_title = "Chair"
    test_input = torch.tensor([encode(test_title)])
    pred_salary = model(test_input).item()
    print(f"Predicted salary for '{test_title}': ${pred_salary:.2f}")